# Day 20 — PyBaMM full-protocol segmented audit

## Purpose

This notebook starts after Day 19A, which closed the retrospective phase and geometry audit.

Day 19A established that the historical PyBaMM DC–AC implementation line from nb04–nb20 was discharge-first relative to the MJ1 experimental charge-first ARB intent. It also established the decomposition

$$
\Delta t_{\mathrm{model}}(Q)
=
\Delta t_{\mathrm{geom}}(Q)
+
\Delta t_{\mathrm{resid}}(Q)
$$

and showed that, in prescribed-current CC simulations, raw $\Delta t(Q)$ is largely explained by current geometry, while the non-geometric residual remains near zero or weak in clean lineages.

## Day 20 question

Day 20 does **not** repeat the phase audit.

The purpose is to examine whether full-protocol PyBaMM simulations contain additional contributions beyond the prescribed-current CC geometry term, especially from:

1. AC-on prescribed-current CC geometry  
2. Voltage-limit event timing  
3. AC-off / control-state transition  
4. CV feedback trajectory  
5. Any remaining non-geometric residual

## Scope

This notebook is simulation-focused.

Primary data source:

- PyBaMM-generated trajectories and audit outputs in this repository

Not primary data source:

- NGU201 MJ1 raw experimental CSV files

Experimental MJ1 data may be used later as an external reference, but this notebook first audits the PyBaMM full-protocol behavior.

## Methodological boundary

The following claims are **not** allowed without segmented evidence:

- “PyBaMM reproduces MJ1 non-geometric state-layer acceleration.”
- “DC–AC acceleration is absent in all full-protocol cases.”
- “Raw $\Delta t(Q)$ is a mechanism-level observable.”

The correct Day 20 objective is to separate current-geometry, voltage-boundary, control-transition, and CV-feedback contributions.

In [2]:
# Cell 1 — Day20 bootstrap: PyBaMM full-protocol segmented audit
#
# Purpose:
#   Start a new audit layer after Day19A closure.
#   Day20 does not modify Day19A outputs.
#
# Scope:
#   Simulation-focused PyBaMM full-protocol audit.
#   Do not load NGU201 / MJ1 raw experimental CSVs in this notebook stage.

from pathlib import Path
from datetime import datetime
import json
import re
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    """
    Walk upward until a directory containing both data/ and notebooks/ is found.
    Works whether the notebook is launched from repo root or notebooks/.
    """
    start = Path.cwd() if start is None else Path(start).resolve()

    for p in [start] + list(start.parents):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p

    raise FileNotFoundError(
        f"Could not find repo root from {start}. "
        "Expected a parent containing data/ and notebooks/."
    )

REPO = find_repo_root()
DATA = REPO / "data"
DOCS = REPO / "docs"
NB_DIR = REPO / "notebooks"

assert DATA.exists(), f"Missing data dir: {DATA}"
assert DOCS.exists(), f"Missing docs dir: {DOCS}"
assert NB_DIR.exists(), f"Missing notebooks dir: {NB_DIR}"

day19_required = [
    DATA / "day19A_step6_evidence_register.csv",
    DATA / "day19A_step6_final_verdict_summary.csv",
    DOCS / "day19A_retrospective_audit.md",
]

for p in day19_required:
    assert p.exists(), f"Missing Day19A closure artifact: {p}"

print("=" * 72)
print("Day20 — PyBaMM full-protocol segmented audit")
print("=" * 72)
print(f"CWD   = {Path.cwd().resolve()}")
print(f"REPO  = {REPO}")
print(f"DATA  = {DATA}")
print(f"DOCS  = {DOCS}")
print(f"NB_DIR= {NB_DIR}")
print(f"Start = {datetime.now().isoformat(timespec='seconds')}")
print()
print("Day19A closure artifacts verified.")
print("Scope: PyBaMM simulation outputs only. NGU201/MJ1 raw experimental CSVs are out of scope for this notebook stage.")

Day20 — PyBaMM full-protocol segmented audit
CWD   = /Users/louislu/pybamm-dcac-superimposed/notebooks
REPO  = /Users/louislu/pybamm-dcac-superimposed
DATA  = /Users/louislu/pybamm-dcac-superimposed/data
DOCS  = /Users/louislu/pybamm-dcac-superimposed/docs
NB_DIR= /Users/louislu/pybamm-dcac-superimposed/notebooks
Start = 2026-05-06T14:35:53

Day19A closure artifacts verified.
Scope: PyBaMM simulation outputs only. NGU201/MJ1 raw experimental CSVs are out of scope for this notebook stage.


In [3]:
# Cell 2 — Inventory PyBaMM full-protocol / voltage-boundary / CV-related outputs
#
# Purpose:
#   Locate existing PyBaMM simulation outputs relevant to Day20 full-protocol
#   segmented audit.
#
# This is NOT an NGU201 / experimental raw CSV search.
#
# Outputs:
#   data/day20_step1_pybamm_full_protocol_output_inventory.csv

import json
import numpy as np
import pandas as pd
from pathlib import Path

patterns = [
    "day18_step1*",
    "day18_step2*",
    "day18B*",
    "day16_step2*",
    "day16_step3a*",
    "*phase_audit*",
    "*anchor*",
    "*trajectory*",
    "*trajectories*",
    "*dt_Q*",
    "*dtQ*",
    "*Vmax*",
    "*Q_to_Vmax*",
    "*metadata*",
]

hits = {}
for pat in patterns:
    for p in DATA.glob(pat):
        if not p.is_file():
            continue
        if p.name.startswith("day19A_"):
            continue
        hits[p.resolve()] = p

rows = []

for p in sorted(hits.values(), key=lambda x: x.name):
    row = {
        "path": str(p.relative_to(REPO)),
        "name": p.name,
        "suffixes": "".join(p.suffixes),
        "size_kb": round(p.stat().st_size / 1024, 2),
        "mtime": pd.Timestamp.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
        "read_ok": False,
        "n_rows": np.nan,
        "n_cols_or_keys": np.nan,
        "columns_or_keys": "",
        "role_guess": "",
        "has_time": False,
        "has_current": False,
        "has_voltage": False,
        "has_Q": False,
        "has_dt": False,
        "has_event": False,
        "has_cv_or_termination": False,
        "has_frequency": False,
        "error": "",
    }

    try:
        if p.suffix.lower() == ".csv" or p.name.endswith(".csv.gz"):
            df_head = pd.read_csv(p, compression="infer", nrows=5)
            cols = list(df_head.columns)
            lc_all = " ".join([p.name.lower()] + [c.lower() for c in cols])

            row["read_ok"] = True
            row["n_cols_or_keys"] = len(cols)
            row["columns_or_keys"] = json.dumps(cols, ensure_ascii=False)

            row["has_time"] = any(s in lc_all for s in ["time", "t_s", "t_dc", "t_dcac", "t_protocol", "t_end"])
            row["has_current"] = any(s in lc_all for s in ["current", "i_", "i_a", "i_py"])
            row["has_voltage"] = any(s in lc_all for s in ["voltage", "vmax", "v_max", "v_min", "v_init"])
            row["has_Q"] = any(s in lc_all for s in ["q_", "q_ah", "q_net", "q_to_vmax", "q_hi", "q_low"])
            row["has_dt"] = any(s in lc_all for s in ["dt", "delta_t", "dtq"])
            row["has_event"] = any(s in lc_all for s in ["event", "q_to_vmax", "t_to_vmax", "vmax"])
            row["has_cv_or_termination"] = any(s in lc_all for s in ["cv", "termination", "cutoff", "t_end", "term_reason"])
            row["has_frequency"] = any(s in lc_all for s in ["f_hz", "freq", "frequency", "t_anchor", "period"])

            if "trajector" in p.name.lower():
                row["role_guess"] = "trajectory_or_cache_csv"
            elif row["has_event"] and row["has_cv_or_termination"]:
                row["role_guess"] = "event_boundary_metadata"
            elif row["has_dt"]:
                row["role_guess"] = "dtQ_curve_or_summary"
            elif row["has_frequency"]:
                row["role_guess"] = "frequency_protocol_design"
            else:
                row["role_guess"] = "csv_auxiliary"

            # row count
            try:
                row["n_rows"] = sum(len(chunk) for chunk in pd.read_csv(p, compression="infer", chunksize=100000))
            except Exception:
                row["n_rows"] = np.nan

        elif p.suffix.lower() == ".npz":
            z = np.load(p, allow_pickle=True)
            keys = list(z.keys())
            lc_all = " ".join([p.name.lower()] + [k.lower() for k in keys])

            row["read_ok"] = True
            row["n_cols_or_keys"] = len(keys)
            row["columns_or_keys"] = json.dumps(keys, ensure_ascii=False)

            row["has_time"] = any(s in lc_all for s in ["time", "t", "t_s"])
            row["has_current"] = any(s in lc_all for s in ["current", "i_", "i_a", "i_py"])
            row["has_voltage"] = any(s in lc_all for s in ["voltage", "v", "v_terminal"])
            row["has_Q"] = any(s in lc_all for s in ["q", "q_net"])
            row["has_dt"] = any(s in lc_all for s in ["dt", "delta"])
            row["has_event"] = any(s in lc_all for s in ["event", "vmax", "q_to_vmax"])
            row["has_cv_or_termination"] = any(s in lc_all for s in ["cv", "termination", "cutoff", "term"])
            row["has_frequency"] = any(s in lc_all for s in ["f_hz", "freq", "period", "tau"])

            row["role_guess"] = "npz_trajectory_cache"

        elif p.suffix.lower() in [".json", ".txt", ".yaml", ".yml"]:
            txt = p.read_text(encoding="utf-8", errors="ignore")[:5000].lower()
            row["read_ok"] = True
            row["columns_or_keys"] = ""

            row["has_time"] = any(s in txt for s in ["time", "t_s", "t_end"])
            row["has_current"] = any(s in txt for s in ["current", "i_py", "i_a"])
            row["has_voltage"] = any(s in txt for s in ["voltage", "vmax", "v_max"])
            row["has_Q"] = any(s in txt for s in ["q_net", "q_to_vmax", "q_hi"])
            row["has_dt"] = any(s in txt for s in ["dt", "delta_t"])
            row["has_event"] = any(s in txt for s in ["event", "vmax", "q_to_vmax"])
            row["has_cv_or_termination"] = any(s in txt for s in ["cv", "termination", "cutoff"])
            row["has_frequency"] = any(s in txt for s in ["f_hz", "frequency", "period", "tau"])

            row["role_guess"] = "text_or_json_metadata"

    except Exception as e:
        row["error"] = str(e)[:500]

    rows.append(row)

inv = pd.DataFrame(rows)

out = DATA / "day20_step1_pybamm_full_protocol_output_inventory.csv"
inv.to_csv(out, index=False)

print(f"Wrote: {out}")
print(f"Candidate files found: {len(inv)}")

if len(inv):
    display_cols = [
        "path", "size_kb", "read_ok", "role_guess",
        "has_time", "has_current", "has_voltage", "has_Q",
        "has_dt", "has_event", "has_cv_or_termination",
        "has_frequency", "columns_or_keys", "error"
    ]
    display(inv[display_cols].sort_values(["role_guess", "path"]).head(120))

print("\nRole guess counts:")
if len(inv):
    print(inv["role_guess"].value_counts(dropna=False).to_string())

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step1_pybamm_full_protocol_output_inventory.csv
Candidate files found: 30


,path,size_kb,read_ok,role_guess,has_time,has_current,has_voltage,has_Q,has_dt,has_event,has_cv_or_termination,has_frequency,columns_or_keys,error
17,data/day18_step1c_first_passage_dV_audit_v3.csv,0.39,True,csv_auxiliary,False,True,True,True,False,False,False,False,"[""param_set"", ""Q_nom_Ah"", ""q_lo_Ah"", ""q_hi_Ah""...",
3,data/day16_step3a_dtQ_curves_long.csv.gz,129.47,True,dtQ_curve_or_summary,True,True,False,True,True,False,False,False,"[""param_set"", ""chem_tag"", ""condition"", ""pair_r...",
4,data/day16_step3a_dtQ_table.csv,13.40,True,dtQ_curve_or_summary,True,True,True,True,True,True,False,False,"[""param_set"", ""chem_tag"", ""condition"", ""pair_r...",
5,data/day18B_protocol_design_full.csv,20.65,True,dtQ_curve_or_summary,False,True,True,True,True,False,True,True,"[""param_set"", ""Q_nom_Ah"", ""tau95_eq_s"", ""V_max...",
6,data/day18B_protocol_design_smoke.csv,7.72,True,dtQ_curve_or_summary,False,True,True,True,True,False,True,True,"[""param_set"", ""Q_nom_Ah"", ""tau95_eq_s"", ""V_max...",
7,data/day18B_smoke_dtQ_resid_curves_long.csv.gz,40.77,True,dtQ_curve_or_summary,False,False,False,True,True,False,False,False,"[""pair_id"", ""param_set"", ""protocol_label"", ""DC...",
8,data/day18B_smoke_dtQ_resid_summary.csv,7.30,True,dtQ_curve_or_summary,False,True,True,True,True,True,False,False,"[""pair_id"", ""param_set"", ""protocol_label"", ""DC...",
20,data/day18_step2_dt_Q_curves_long_v3.csv.gz,27.42,True,dtQ_curve_or_summary,True,False,False,True,True,False,False,False,"[""param_set"", ""anchor_label"", ""Q_Ah"", ""Q_frac_...",
21,data/day18_step2_dt_Q_curves_long_v4_charge_fi...,16.06,True,dtQ_curve_or_summary,True,False,False,True,True,False,False,False,"[""param_set"", ""anchor_label"", ""phase_label"", ""...",
22,data/day18_step2_dt_Q_v3_vs_v4_common_window.csv,0.75,True,dtQ_curve_or_summary,False,True,False,True,True,False,False,False,"[""param_set"", ""v3_status"", ""common_q_hi_Ah"", ""...",



Role guess counts:
role_guess
dtQ_curve_or_summary       16
event_boundary_metadata     8
npz_trajectory_cache        4
trajectory_or_cache_csv     1
csv_auxiliary               1


In [4]:
# Cell 3 — Inspect Day18 v4 charge-first trajectory schema
#
# Purpose:
#   Inspect the main PyBaMM full-protocol trajectory cache for Day20:
#       data/day18_step1_phase_audit_trajectories_v4_charge_first.npz
#
# Outputs:
#   data/day20_step1_day18_v4_trajectory_schema.csv
#
# Goal:
#   Determine which fields are available for segmented audit:
#     - time
#     - Q_net
#     - voltage
#     - current, if available
#     - protocol / param-set naming structure

import json
import numpy as np
import pandas as pd
from pathlib import Path

TRAJ_V4 = DATA / "day18_step1_phase_audit_trajectories_v4_charge_first.npz"
assert TRAJ_V4.exists(), f"Missing: {TRAJ_V4}"

z = np.load(TRAJ_V4, allow_pickle=True)
keys = list(z.keys())

print(f"Loaded: {TRAJ_V4}")
print(f"n_keys = {len(keys)}")

rows = []

for k in keys:
    arr = np.asarray(z[k])
    kl = k.lower()

    # Expected naming: <param_set>__<protocol>__<field>
    parts = k.split("__")
    if len(parts) >= 3:
        param_set = parts[0]
        protocol = parts[1]
        field = "__".join(parts[2:])
    elif len(parts) == 2:
        param_set = parts[0]
        protocol = ""
        field = parts[1]
    else:
        param_set = ""
        protocol = ""
        field = k

    role = "unknown"
    if field.lower() in ["t", "t_s", "time", "time_s"]:
        role = "time"
    elif "q" in field.lower():
        role = "Q"
    elif field.lower().startswith("v") or "voltage" in field.lower():
        role = "voltage"
    elif field.lower().startswith("i") or "current" in field.lower():
        role = "current"
    elif "soc" in field.lower():
        role = "soc"

    finite = np.isfinite(arr).all() if np.issubdtype(arr.dtype, np.number) else False

    rows.append({
        "key": k,
        "param_set": param_set,
        "protocol": protocol,
        "field": field,
        "role": role,
        "shape": str(arr.shape),
        "dtype": str(arr.dtype),
        "size": int(arr.size),
        "finite_all": bool(finite),
        "min": float(np.nanmin(arr)) if arr.size and np.issubdtype(arr.dtype, np.number) else np.nan,
        "max": float(np.nanmax(arr)) if arr.size and np.issubdtype(arr.dtype, np.number) else np.nan,
    })

schema_df = pd.DataFrame(rows)

out = DATA / "day20_step1_day18_v4_trajectory_schema.csv"
schema_df.to_csv(out, index=False)

print(f"Wrote: {out}")

print("\nRole counts:")
print(schema_df["role"].value_counts(dropna=False).to_string())

print("\nParam/protocol/field preview:")
display(
    schema_df[
        ["key", "param_set", "protocol", "field", "role", "shape", "min", "max"]
    ].head(80)
)

print("\nProtocol coverage:")
coverage = (
    schema_df
    .groupby(["param_set", "protocol"])["role"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="roles")
)
display(coverage)

Loaded: /Users/louislu/pybamm-dcac-superimposed/data/day18_step1_phase_audit_trajectories_v4_charge_first.npz
n_keys = 24
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step1_day18_v4_trajectory_schema.csv

Role counts:
role
time       6
voltage    6
current    6
Q          6

Param/protocol/field preview:


,key,param_set,protocol,field,role,shape,min,max
0,Chen2020__DC_0p2C__t,Chen2020,DC_0p2C,t,time,"(9464,)",0.000000,16659.715676
1,Chen2020__DC_0p2C__V,Chen2020,DC_0p2C,V,voltage,"(9464,)",3.157897,4.200000
2,Chen2020__DC_0p2C__I,Chen2020,DC_0p2C,I,current,"(9464,)",-1.000000,-1.000000
3,Chen2020__DC_0p2C__Q_net,Chen2020,DC_0p2C,Q_net,Q,"(9464,)",0.000000,4.627699
4,Chen2020__DCAC_DC0p2C_AC0p5C_10tau__t,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,t,time,"(8964,)",0.000000,12799.119672
5,Chen2020__DCAC_DC0p2C_AC0p5C_10tau__V,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,V,voltage,"(8964,)",3.157897,4.200000
6,Chen2020__DCAC_DC0p2C_AC0p5C_10tau__I,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,I,current,"(8964,)",-3.500000,1.500000
7,Chen2020__DCAC_DC0p2C_AC0p5C_10tau__Q_net,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,Q_net,Q,"(8964,)",0.000000,3.770074
8,OKane2022__DC_0p2C__t,OKane2022,DC_0p2C,t,time,"(9403,)",0.000000,16548.641555
9,OKane2022__DC_0p2C__V,OKane2022,DC_0p2C,V,voltage,"(9403,)",3.177915,4.200000



Protocol coverage:


,param_set,protocol,roles
0,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,"[Q, current, time, voltage]"
1,Chen2020,DC_0p2C,"[Q, current, time, voltage]"
2,OKane2022,DCAC_DC0p2C_AC0p5C_10tau,"[Q, current, time, voltage]"
3,OKane2022,DC_0p2C,"[Q, current, time, voltage]"
4,ORegan2022,DCAC_DC0p2C_AC0p5C_10tau,"[Q, current, time, voltage]"
5,ORegan2022,DC_0p2C,"[Q, current, time, voltage]"


In [6]:
# Cell 4 — Day18 v4 charge-first event-boundary and segment audit
#
# Purpose:
#   Use trajectory-level t, V, I, Q_net arrays from
#       day18_step1_phase_audit_trajectories_v4_charge_first.npz
#   and metadata from
#       day18_step1_phase_audit_matrix_v4_charge_first.csv
#       day18_step2_dt_Q_audit_v4_charge_first.csv
#
#   to define available PyBaMM full-protocol / voltage-boundary segments.
#
# Important scope note:
#   The available v4 trajectory cache appears to be voltage-limit / phase-audit
#   trajectory data, not a full CC+CV trajectory with CV current decay.
#   This cell therefore audits event-boundary segmentation first.
#
# Outputs:
#   data/day20_step2_v4_event_boundary_audit.csv
#   data/day20_step2_v4_segment_boundaries.csv

import numpy as np
import pandas as pd
from pathlib import Path

TRAJ_V4 = DATA / "day18_step1_phase_audit_trajectories_v4_charge_first.npz"
MATRIX_V4 = DATA / "day18_step1_phase_audit_matrix_v4_charge_first.csv"
AUDIT_V4 = DATA / "day18_step2_dt_Q_audit_v4_charge_first.csv"

assert TRAJ_V4.exists(), f"Missing: {TRAJ_V4}"
assert MATRIX_V4.exists(), f"Missing: {MATRIX_V4}"
assert AUDIT_V4.exists(), f"Missing: {AUDIT_V4}"

z = np.load(TRAJ_V4, allow_pickle=True)
matrix = pd.read_csv(MATRIX_V4)
audit = pd.read_csv(AUDIT_V4)

print(f"Loaded trajectory cache: {TRAJ_V4.name}")
print(f"Loaded matrix metadata : {MATRIX_V4.name} shape={matrix.shape}")
print(f"Loaded dtQ audit table : {AUDIT_V4.name} shape={audit.shape}")


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def get_traj(param_set, protocol):
    """
    Extract t, V, I, Q arrays for one param_set / protocol.
    Expected keys:
        <param_set>__<protocol>__t
        <param_set>__<protocol>__V
        <param_set>__<protocol>__I
        <param_set>__<protocol>__Q_net
    Q_net is in Ah in the v4 trajectory cache.
    """
    keys = {
        "t": f"{param_set}__{protocol}__t",
        "V": f"{param_set}__{protocol}__V",
        "I": f"{param_set}__{protocol}__I",
        "Q": f"{param_set}__{protocol}__Q_net",
    }

    missing = [k for k in keys.values() if k not in z.files]
    if missing:
        raise KeyError(f"Missing trajectory keys for {param_set}/{protocol}: {missing}")

    out = {
        "t_s": np.asarray(z[keys["t"]], dtype=float),
        "V_V": np.asarray(z[keys["V"]], dtype=float),
        "I_A": np.asarray(z[keys["I"]], dtype=float),
        "Q_Ah": np.asarray(z[keys["Q"]], dtype=float),
    }

    n = len(out["t_s"])
    for name, arr in out.items():
        if len(arr) != n:
            raise ValueError(f"{param_set}/{protocol}: length mismatch for {name}: {len(arr)} vs {n}")
        if not np.all(np.isfinite(arr)):
            raise ValueError(f"{param_set}/{protocol}: non-finite values in {name}")

    if n > 1 and not np.all(np.diff(out["t_s"]) > 0):
        raise ValueError(f"{param_set}/{protocol}: t_s not strictly increasing")

    return out


def first_crossing_event(t_s, y, q_Ah, threshold):
    """
    First y >= threshold event with linear interpolation in t and Q.
    Returns NaN fields if no crossing.
    """
    t_s = np.asarray(t_s, dtype=float)
    y = np.asarray(y, dtype=float)
    q_Ah = np.asarray(q_Ah, dtype=float)

    crossed = y >= threshold

    if not crossed.any():
        return {
            "event_found": False,
            "event_idx": np.nan,
            "t_event_s": np.nan,
            "Q_event_Ah": np.nan,
            "y_event": np.nan,
        }

    idx = int(np.argmax(crossed))

    if idx == 0:
        return {
            "event_found": True,
            "event_idx": idx,
            "t_event_s": float(t_s[0]),
            "Q_event_Ah": float(q_Ah[0]),
            "y_event": float(y[0]),
        }

    y0, y1 = y[idx - 1], y[idx]
    t0, t1 = t_s[idx - 1], t_s[idx]
    q0, q1 = q_Ah[idx - 1], q_Ah[idx]

    if y1 == y0:
        frac = 1.0
    else:
        frac = (threshold - y0) / (y1 - y0)
        frac = float(np.clip(frac, 0.0, 1.0))

    return {
        "event_found": True,
        "event_idx": idx,
        "t_event_s": float(t0 + frac * (t1 - t0)),
        "Q_event_Ah": float(q0 + frac * (q1 - q0)),
        "y_event": float(threshold),
    }


def trajectory_summary(param_set, protocol, traj, meta_row):
    """
    Summarize one trajectory and compare event crossing with stored metadata.
    """
    Vmax_cutoff = float(meta_row["V_max_cutoff"])
    event = first_crossing_event(
        t_s=traj["t_s"],
        y=traj["V_V"],
        q_Ah=traj["Q_Ah"],
        threshold=Vmax_cutoff,
    )

    t_meta = float(meta_row["t_to_Vmax_s"]) if pd.notna(meta_row["t_to_Vmax_s"]) else np.nan
    q_meta = float(meta_row["Q_to_Vmax_Ah"]) if pd.notna(meta_row["Q_to_Vmax_Ah"]) else np.nan

    post_mask = traj["t_s"] > event["t_event_s"] + 1e-9 if event["event_found"] else np.zeros_like(traj["t_s"], dtype=bool)
    n_post = int(post_mask.sum())

    # Heuristic: a true CV tail would have nontrivial samples after voltage event
    # and current drift after the event. In the v4 phase-audit cache this is expected false.
    if n_post >= 5:
        I_post = traj["I_A"][post_mask]
        I_post_range = float(np.nanmax(I_post) - np.nanmin(I_post))
    else:
        I_post_range = np.nan

    has_post_event_tail = n_post >= 5
    has_cv_like_current_decay = bool(has_post_event_tail and np.isfinite(I_post_range) and I_post_range > 0.05)

    return {
        "param_set": param_set,
        "protocol": protocol,
        "phase_label": meta_row.get("phase_label", np.nan),
        "anchor_label": meta_row.get("anchor_label", np.nan),

        "n_samples": len(traj["t_s"]),
        "t_start_s": float(traj["t_s"][0]),
        "t_end_s": float(traj["t_s"][-1]),
        "Q_start_Ah": float(traj["Q_Ah"][0]),
        "Q_end_Ah": float(traj["Q_Ah"][-1]),
        "V_min_V": float(np.nanmin(traj["V_V"])),
        "V_max_V": float(np.nanmax(traj["V_V"])),
        "I_min_A": float(np.nanmin(traj["I_A"])),
        "I_max_A": float(np.nanmax(traj["I_A"])),
        "I_initial_A": float(traj["I_A"][0]),
        "I_final_A": float(traj["I_A"][-1]),

        "V_max_cutoff": Vmax_cutoff,
        "event_found_calc": event["event_found"],
        "event_idx_calc": event["event_idx"],
        "t_to_Vmax_calc_s": event["t_event_s"],
        "Q_to_Vmax_calc_Ah": event["Q_event_Ah"],
        "t_to_Vmax_meta_s": t_meta,
        "Q_to_Vmax_meta_Ah": q_meta,
        "t_to_Vmax_err_s": event["t_event_s"] - t_meta if np.isfinite(t_meta) else np.nan,
        "Q_to_Vmax_err_Ah": event["Q_event_Ah"] - q_meta if np.isfinite(q_meta) else np.nan,

        "n_post_event_samples": n_post,
        "has_post_event_tail": has_post_event_tail,
        "I_post_event_range_A": I_post_range,
        "has_cv_like_current_decay": has_cv_like_current_decay,

        "termination_raw": meta_row.get("termination_raw", np.nan),
        "feasibility_verdict": meta_row.get("feasibility_verdict", np.nan),
        "frac_below_Vmin": meta_row.get("frac_below_Vmin", np.nan),
        "frac_above_Vmax": meta_row.get("frac_above_Vmax", np.nan),
    }


# ---------------------------------------------------------------------
# Build event-boundary audit table
# ---------------------------------------------------------------------

required_matrix_cols = {
    "param_set", "protocol", "phase_label", "anchor_label",
    "V_max_cutoff", "Q_to_Vmax_Ah", "t_to_Vmax_s",
    "termination_raw", "feasibility_verdict",
}
missing = required_matrix_cols - set(matrix.columns)
assert not missing, f"Missing matrix columns: {missing}"

event_rows = []

# Use only protocols present in the v4 trajectory cache
schema_path = DATA / "day20_step1_day18_v4_trajectory_schema.csv"
schema = pd.read_csv(schema_path)
coverage = schema.groupby(["param_set", "protocol"])["role"].apply(lambda x: set(x)).reset_index()

for _, row in coverage.iterrows():
    param_set = row["param_set"]
    protocol = row["protocol"]
    roles = row["role"]

    if not {"time", "voltage", "current", "Q"}.issubset(roles):
        continue

    m = matrix[(matrix["param_set"] == param_set) & (matrix["protocol"] == protocol)]
    if len(m) != 1:
        raise ValueError(f"Expected one metadata row for {param_set}/{protocol}, got {len(m)}")

    meta_row = m.iloc[0]
    traj = get_traj(param_set, protocol)
    event_rows.append(trajectory_summary(param_set, protocol, traj, meta_row))

event_df = pd.DataFrame(event_rows)

out_event = DATA / "day20_step2_v4_event_boundary_audit.csv"
event_df.to_csv(out_event, index=False)

print(f"Wrote: {out_event}")
display(event_df)


# ---------------------------------------------------------------------
# Pair-level segment boundaries
# ---------------------------------------------------------------------

audit_required = {"param_set", "anchor_label", "phase_label", "f_anchor_Hz", "q_lo_full_Ah", "q_lo_stable_added_Ah", "q_hi_Ah"}
missing_audit = audit_required - set(audit.columns)
assert not missing_audit, f"Missing audit columns: {missing_audit}"

segment_rows = []

for _, ar in audit.iterrows():
    param_set = ar["param_set"]
    anchor_label = ar["anchor_label"]
    phase_label = ar["phase_label"]

    pair_meta = matrix[
        (matrix["param_set"] == param_set)
        & (matrix["anchor_label"] == anchor_label)
        & (matrix["phase_label"] == phase_label)
    ]

    dc_rows = pair_meta[pair_meta["protocol"].str.startswith("DC_")]
    dcac_rows = pair_meta[pair_meta["protocol"].str.startswith("DCAC_")]

    if len(dc_rows) != 1 or len(dcac_rows) != 1:
        raise ValueError(
            f"Expected one DC and one DCAC row for {param_set}/{anchor_label}, "
            f"got DC={len(dc_rows)}, DCAC={len(dcac_rows)}"
        )

    dc = dc_rows.iloc[0]
    dcac = dcac_rows.iloc[0]

    Q_vmax_DC = float(dc["Q_to_Vmax_Ah"])
    Q_vmax_DCAC = float(dcac["Q_to_Vmax_Ah"])
    t_vmax_DC = float(dc["t_to_Vmax_s"])
    t_vmax_DCAC = float(dcac["t_to_Vmax_s"])

    q_A_lo = float(ar["q_lo_stable_added_Ah"])
    q_A_hi = float(ar["q_hi_Ah"])

    # Segment B is meaningful when DCAC reaches Vmax at lower Q than DC.
    q_B_lo = min(Q_vmax_DCAC, Q_vmax_DC)
    q_B_hi = max(Q_vmax_DCAC, Q_vmax_DC)
    boundary_span_Ah = q_B_hi - q_B_lo

    # If the stored stable Q window ends below the earliest Vmax, then A corresponds to
    # prescribed-current shared window and B is outside the stored dt(Q) window.
    A_before_earliest_event = q_A_hi <= min(Q_vmax_DC, Q_vmax_DCAC) + 1e-9

    segment_rows.append({
        "param_set": param_set,
        "anchor_label": anchor_label,
        "phase_label": phase_label,
        "protocol_DC": dc["protocol"],
        "protocol_DCAC": dcac["protocol"],

        "f_anchor_Hz": float(ar["f_anchor_Hz"]),
        "Q_nom_Ah": float(ar["Q_nom_Ah"]),
        "T_period_s": float(ar["T_period_s"]),

        "Vmax_cutoff_DC": float(dc["V_max_cutoff"]),
        "Vmax_cutoff_DCAC": float(dcac["V_max_cutoff"]),

        "Q_to_Vmax_DC_Ah": Q_vmax_DC,
        "Q_to_Vmax_DCAC_Ah": Q_vmax_DCAC,
        "Q_to_Vmax_shift_Ah": Q_vmax_DC - Q_vmax_DCAC,
        "Q_to_Vmax_shift_pct_nom": (Q_vmax_DC - Q_vmax_DCAC) / float(ar["Q_nom_Ah"]) * 100.0,

        "t_to_Vmax_DC_s": t_vmax_DC,
        "t_to_Vmax_DCAC_s": t_vmax_DCAC,
        "t_to_Vmax_shift_s": t_vmax_DC - t_vmax_DCAC,

        "segment_A_name": "AC-on prescribed-current shared dt(Q) window",
        "segment_A_Q_lo_Ah": q_A_lo,
        "segment_A_Q_hi_Ah": q_A_hi,
        "segment_A_before_earliest_Vmax": A_before_earliest_event,

        "segment_B_name": "voltage-boundary event separation interval",
        "segment_B_Q_lo_Ah": q_B_lo,
        "segment_B_Q_hi_Ah": q_B_hi,
        "segment_B_span_Ah": boundary_span_Ah,
        "segment_B_available_in_current_npz": False,

        "has_cv_feedback_in_current_npz": bool(
            event_df[
                (event_df["param_set"] == param_set)
                & (event_df["protocol"].isin([dc["protocol"], dcac["protocol"]]))
            ]["has_cv_like_current_decay"].any()
        ),

        "day20_scope_for_this_pair": (
            "CC/event-boundary audit only; CV feedback requires additional full CC+CV trajectory"
        ),
    })

segment_df = pd.DataFrame(segment_rows)

out_segment = DATA / "day20_step2_v4_segment_boundaries.csv"
segment_df.to_csv(out_segment, index=False)

print(f"\nWrote: {out_segment}")
display(segment_df)

print("\nBoundary shift summary:")
display(
    segment_df[
        [
            "param_set",
            "Q_to_Vmax_DC_Ah",
            "Q_to_Vmax_DCAC_Ah",
            "Q_to_Vmax_shift_Ah",
            "Q_to_Vmax_shift_pct_nom",
            "t_to_Vmax_shift_s",
            "segment_A_before_earliest_Vmax",
            "has_cv_feedback_in_current_npz",
        ]
    ]
)

# ---------------------------------------------------------------------
# Validation checks
# ---------------------------------------------------------------------
# Metadata event locations are authoritative. The trajectory-derived event
# time is a sampled-trajectory reconstruction and can differ by several
# seconds near solver termination. We therefore use a relaxed time tolerance
# and a tighter Q tolerance.

T_EVENT_RECON_TOL_S = 15.0
Q_EVENT_RECON_TOL_AH = 5e-3

event_df["event_time_reconstruction_ok"] = event_df["t_to_Vmax_err_s"].abs() < T_EVENT_RECON_TOL_S
event_df["event_Q_reconstruction_ok"] = event_df["Q_to_Vmax_err_Ah"].abs() < Q_EVENT_RECON_TOL_AH

# Re-save event table with validation flags
event_df.to_csv(out_event, index=False)

print("\nEvent reconstruction validation:")
print(
    event_df[
        [
            "param_set", "protocol",
            "t_to_Vmax_calc_s", "t_to_Vmax_meta_s", "t_to_Vmax_err_s",
            "Q_to_Vmax_calc_Ah", "Q_to_Vmax_meta_Ah", "Q_to_Vmax_err_Ah",
            "event_time_reconstruction_ok",
            "event_Q_reconstruction_ok",
        ]
    ].to_string(index=False)
)

assert event_df["event_found_calc"].all(), (
    "At least one trajectory did not reach Vmax cutoff."
)

assert event_df["event_time_reconstruction_ok"].all(), (
    f"Trajectory-derived t_to_Vmax differs from metadata by >{T_EVENT_RECON_TOL_S} s:\n"
    f"{event_df[~event_df['event_time_reconstruction_ok']][['param_set', 'protocol', 't_to_Vmax_calc_s', 't_to_Vmax_meta_s', 't_to_Vmax_err_s']]}"
)

assert event_df["event_Q_reconstruction_ok"].all(), (
    f"Trajectory-derived Q_to_Vmax differs from metadata by >{Q_EVENT_RECON_TOL_AH} Ah:\n"
    f"{event_df[~event_df['event_Q_reconstruction_ok']][['param_set', 'protocol', 'Q_to_Vmax_calc_Ah', 'Q_to_Vmax_meta_Ah', 'Q_to_Vmax_err_Ah']]}"
)

assert not event_df["has_cv_like_current_decay"].any(), (
    "Unexpected CV-like current decay found in current v4 trajectory cache."
)

print("\n" + "=" * 72)
print("Cell 4 PASSED — Day18 v4 event-boundary audit complete")
print("Scope confirmed: CC/event-boundary only; no CV feedback trajectory in current NPZ.")
print("=" * 72)

Loaded trajectory cache: day18_step1_phase_audit_trajectories_v4_charge_first.npz
Loaded matrix metadata : day18_step1_phase_audit_matrix_v4_charge_first.csv shape=(6, 36)
Loaded dtQ audit table : day18_step2_dt_Q_audit_v4_charge_first.csv shape=(3, 71)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step2_v4_event_boundary_audit.csv


,param_set,protocol,phase_label,anchor_label,n_samples,t_start_s,t_end_s,Q_start_Ah,Q_end_Ah,V_min_V,...,t_to_Vmax_err_s,Q_to_Vmax_err_Ah,n_post_event_samples,has_post_event_tail,I_post_event_range_A,has_cv_like_current_decay,termination_raw,feasibility_verdict,frac_below_Vmin,frac_above_Vmax
0,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,charge_first,Route2_AC0p5C_10tau_native_charge_first,8964,0.0,12799.119672,0.0,3.770074,3.157897,...,1.181164e+00,1.130151e-03,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000223
1,Chen2020,DC_0p2C,charge_first,Route2_AC0p5C_10tau_native_charge_first,9464,0.0,16659.715676,0.0,4.627699,3.157897,...,9.715676e+00,2.698799e-03,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000634
2,OKane2022,DCAC_DC0p2C_AC0p5C_10tau,charge_first,Route2_AC0p5C_10tau_native_charge_first,9164,0.0,13045.766501,0.0,3.790253,3.177915,...,3.759816e+00,3.430599e-03,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000327
3,OKane2022,DC_0p2C,charge_first,Route2_AC0p5C_10tau_native_charge_first,9403,0.0,16548.641555,0.0,4.596845,3.177915,...,8.641555e+00,2.400432e-03,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000638
4,ORegan2022,DCAC_DC0p2C_AC0p5C_10tau,charge_first,Route2_AC0p5C_10tau_native_charge_first,7798,0.0,9925.854020,0.0,3.210457,3.099282,...,-3.637979e-12,-8.881784e-16,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000128
5,ORegan2022,DC_0p2C,charge_first,Route2_AC0p5C_10tau_native_charge_first,12378,0.0,17006.861901,0.0,4.724128,3.099282,...,0.000000e+00,0.000000e+00,0,False,NaN,False,event: maximum voltage [v],feasible_clean,0.0,0.000081



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step2_v4_segment_boundaries.csv


,param_set,anchor_label,phase_label,protocol_DC,protocol_DCAC,f_anchor_Hz,Q_nom_Ah,T_period_s,Vmax_cutoff_DC,Vmax_cutoff_DCAC,...,segment_A_Q_lo_Ah,segment_A_Q_hi_Ah,segment_A_before_earliest_Vmax,segment_B_name,segment_B_Q_lo_Ah,segment_B_Q_hi_Ah,segment_B_span_Ah,segment_B_available_in_current_npz,has_cv_feedback_in_current_npz,day20_scope_for_this_pair
0,Chen2020,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,0.000408,5.0,2453.513699,4.2,4.2,...,1.0,3.668944,True,voltage-boundary event separation interval,3.768944,4.625000,0.856056,False,False,CC/event-boundary audit only; CV feedback requ...
1,OKane2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,0.000397,5.0,2516.435063,4.2,4.2,...,1.0,3.686822,True,voltage-boundary event separation interval,3.786822,4.594444,0.807622,False,False,CC/event-boundary audit only; CV feedback requ...
2,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,0.000333,5.0,2998.832183,4.4,4.4,...,1.0,3.110457,True,voltage-boundary event separation interval,3.210457,4.724128,1.513671,False,False,CC/event-boundary audit only; CV feedback requ...



Boundary shift summary:


,param_set,Q_to_Vmax_DC_Ah,Q_to_Vmax_DCAC_Ah,Q_to_Vmax_shift_Ah,Q_to_Vmax_shift_pct_nom,t_to_Vmax_shift_s,segment_A_before_earliest_Vmax,has_cv_feedback_in_current_npz
0,Chen2020,4.625000,3.768944,0.856056,17.121118,3852.061493,True,False
1,OKane2022,4.594444,3.786822,0.807622,16.152443,3497.993316,True,False
2,ORegan2022,4.724128,3.210457,1.513671,30.273420,7081.007881,True,False



Event reconstruction validation:
 param_set                 protocol  t_to_Vmax_calc_s  t_to_Vmax_meta_s  t_to_Vmax_err_s  Q_to_Vmax_calc_Ah  Q_to_Vmax_meta_Ah  Q_to_Vmax_err_Ah  event_time_reconstruction_ok  event_Q_reconstruction_ok
  Chen2020 DCAC_DC0p2C_AC0p5C_10tau      12799.119672      12797.938507     1.181164e+00           3.770074           3.768944      1.130151e-03                          True                       True
  Chen2020                  DC_0p2C      16659.715676      16650.000000     9.715676e+00           4.627699           4.625000      2.698799e-03                          True                       True
 OKane2022 DCAC_DC0p2C_AC0p5C_10tau      13045.766501      13042.006684     3.759816e+00           3.790253           3.786822      3.430599e-03                          True                       True
 OKane2022                  DC_0p2C      16548.641555      16540.000000     8.641555e+00           4.596845           4.594444      2.400432e-03              

In [7]:
# Cell 5 — Segment A geometry / residual audit for Day18 v4 charge-first
#
# Purpose:
#   Audit Segment A:
#       AC-on prescribed-current shared-Q window before earliest Vmax event
#
# Inputs:
#   data/day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz
#   data/day18_step2_dt_Q_audit_v4_charge_first.csv
#   data/day20_step2_v4_segment_boundaries.csv
#
# Outputs:
#   data/day20_step3_segment_A_geometry_residual_curves.csv
#   data/day20_step3_segment_A_geometry_residual_summary.csv
#
# Interpretation:
#   If Segment A residual remains near zero, then the charge-first v4
#   prescribed-current shared-Q region is geometry-dominated.
#
# Scope boundary:
#   This does not audit voltage-boundary interval, AC-off transition, or CV feedback.

import numpy as np
import pandas as pd

CURVES_V4 = DATA / "day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz"
AUDIT_V4 = DATA / "day18_step2_dt_Q_audit_v4_charge_first.csv"
SEGMENTS = DATA / "day20_step2_v4_segment_boundaries.csv"

assert CURVES_V4.exists(), f"Missing: {CURVES_V4}"
assert AUDIT_V4.exists(), f"Missing: {AUDIT_V4}"
assert SEGMENTS.exists(), f"Missing: {SEGMENTS}"

curves = pd.read_csv(CURVES_V4, compression="gzip")
audit = pd.read_csv(AUDIT_V4)
segments = pd.read_csv(SEGMENTS)

required_curves = {
    "param_set", "anchor_label", "phase_label",
    "Q_Ah", "t_DC_s", "t_DCAC_s",
    "dtQ_s", "dtQ_geom_s", "dtQ_resid_s",
    "window_full", "window_stable_added",
    "protocol_DC", "protocol_DCAC", "status",
}
missing_curves = required_curves - set(curves.columns)
assert not missing_curves, f"Curves missing columns: {missing_curves}"

required_segments = {
    "param_set", "anchor_label", "phase_label",
    "protocol_DC", "protocol_DCAC",
    "segment_A_Q_lo_Ah", "segment_A_Q_hi_Ah",
    "segment_A_before_earliest_Vmax",
    "Q_to_Vmax_DC_Ah", "Q_to_Vmax_DCAC_Ah",
    "Q_to_Vmax_shift_Ah", "t_to_Vmax_shift_s",
    "has_cv_feedback_in_current_npz",
}
missing_segments = required_segments - set(segments.columns)
assert not missing_segments, f"Segments missing columns: {missing_segments}"

required_audit = {
    "param_set", "anchor_label", "phase_label",
    "Q_nom_Ah", "f_anchor_Hz",
    "full_geom_mean_s", "full_resid_mean_s", "full_resid_max_abs_s",
    "stable_geom_mean_s", "stable_resid_mean_s", "stable_resid_max_abs_s",
}
missing_audit = required_audit - set(audit.columns)
assert not missing_audit, f"Audit table missing columns: {missing_audit}"


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def as_bool_series(s):
    """
    Robustly convert bool-like column values to bool.
    Handles True/False, 1/0, and string variants.
    """
    if s.dtype == bool:
        return s

    return (
        s.astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
        })
        .fillna(False)
        .astype(bool)
    )


def sign_topology(x, threshold_s=1.0):
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)

    if not finite.any():
        return "invalid"

    xx = x[finite]
    pos = np.any(xx > threshold_s)
    neg = np.any(xx < -threshold_s)

    if pos and neg:
        return "mixed"
    if pos:
        return "positive_only"
    if neg:
        return "negative_only"
    return "near_zero"


def safe_abs_ratio(num, den, eps=1e-12):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)

    out = np.full_like(num, np.nan, dtype=float)
    mask = np.isfinite(num) & np.isfinite(den) & (np.abs(den) > eps)
    out[mask] = np.abs(num[mask]) / np.abs(den[mask])
    return out


def segment_A_classification(max_abs_resid, p95_abs_resid, median_abs_resid):
    if not np.isfinite(max_abs_resid):
        return "invalid"
    if max_abs_resid < 0.1 and p95_abs_resid < 0.01:
        return "numerical_null"
    if max_abs_resid < 1.0:
        return "near_zero"
    if p95_abs_resid < 1.0:
        return "spiky_but_stable_near_zero"
    if p95_abs_resid < 10.0:
        return "weak_residual"
    return "requires_inspection"


# ---------------------------------------------------------------------
# Join curves with segment boundaries and audit metadata
# ---------------------------------------------------------------------

join_keys = ["param_set", "anchor_label", "phase_label", "protocol_DC", "protocol_DCAC"]

seg_cols = join_keys + [
    "segment_A_Q_lo_Ah", "segment_A_Q_hi_Ah",
    "segment_A_before_earliest_Vmax",
    "Q_to_Vmax_DC_Ah", "Q_to_Vmax_DCAC_Ah",
    "Q_to_Vmax_shift_Ah", "Q_to_Vmax_shift_pct_nom",
    "t_to_Vmax_shift_s",
    "has_cv_feedback_in_current_npz",
]

segments_meta = segments[seg_cols].drop_duplicates(join_keys)

curves_join = curves.merge(
    segments_meta,
    on=join_keys,
    how="left",
    validate="many_to_one",
    indicator=True,
)

missing_join = curves_join[curves_join["_merge"] != "both"]
assert missing_join.empty, (
    "Some curves failed to join segment boundaries:\n"
    f"{missing_join[join_keys].drop_duplicates().head(20)}"
)

curves_join = curves_join.drop(columns=["_merge"])

audit_meta_cols = [
    "param_set", "anchor_label", "phase_label",
    "Q_nom_Ah", "f_anchor_Hz", "T_period_s",
    "full_geom_mean_s", "full_resid_mean_s", "full_resid_max_abs_s",
    "stable_geom_mean_s", "stable_resid_mean_s", "stable_resid_max_abs_s",
]

audit_meta = audit[audit_meta_cols].drop_duplicates(["param_set", "anchor_label", "phase_label"])

curves_join = curves_join.merge(
    audit_meta,
    on=["param_set", "anchor_label", "phase_label"],
    how="left",
    validate="many_to_one",
)

# Numeric coercion
num_cols = [
    "Q_Ah", "t_DC_s", "t_DCAC_s",
    "dtQ_s", "dtQ_geom_s", "dtQ_resid_s",
    "segment_A_Q_lo_Ah", "segment_A_Q_hi_Ah",
    "Q_to_Vmax_DC_Ah", "Q_to_Vmax_DCAC_Ah",
    "Q_to_Vmax_shift_Ah", "t_to_Vmax_shift_s",
    "Q_nom_Ah", "f_anchor_Hz", "T_period_s",
]
for c in num_cols:
    curves_join[c] = pd.to_numeric(curves_join[c], errors="coerce")

curves_join["window_full_bool"] = as_bool_series(curves_join["window_full"])
curves_join["window_stable_added_bool"] = as_bool_series(curves_join["window_stable_added"])

curves_join["inside_segment_A_by_boundary"] = (
    (curves_join["Q_Ah"] >= curves_join["segment_A_Q_lo_Ah"] - 1e-12)
    & (curves_join["Q_Ah"] <= curves_join["segment_A_Q_hi_Ah"] + 1e-12)
)

curves_join["segment_A_row"] = (
    curves_join["inside_segment_A_by_boundary"]
    & curves_join["window_stable_added_bool"]
)

# Use segment_A_row as primary filter.
segA = curves_join[curves_join["segment_A_row"]].copy()

assert len(segA) > 0, "No Segment A rows found."

# ---------------------------------------------------------------------
# Row-level diagnostics
# ---------------------------------------------------------------------

segA["dtQ_identity_err_s"] = segA["dtQ_s"] - (
    segA["dtQ_geom_s"] + segA["dtQ_resid_s"]
)

segA["abs_resid_s"] = segA["dtQ_resid_s"].abs()
segA["abs_geom_s"] = segA["dtQ_geom_s"].abs()
segA["abs_model_s"] = segA["dtQ_s"].abs()
segA["resid_over_geom"] = safe_abs_ratio(segA["dtQ_resid_s"], segA["dtQ_geom_s"])
segA["resid_over_model"] = safe_abs_ratio(segA["dtQ_resid_s"], segA["dtQ_s"])

# ---------------------------------------------------------------------
# Summary per param_set / anchor
# ---------------------------------------------------------------------

summary_rows = []

for keys, g in segA.groupby(["param_set", "anchor_label", "phase_label"], sort=False):
    param_set, anchor_label, phase_label = keys

    dt_model = g["dtQ_s"].to_numpy(dtype=float)
    dt_geom = g["dtQ_geom_s"].to_numpy(dtype=float)
    dt_resid = g["dtQ_resid_s"].to_numpy(dtype=float)
    abs_resid = np.abs(dt_resid)

    resid_over_geom = g["resid_over_geom"].to_numpy(dtype=float)
    resid_over_model = g["resid_over_model"].to_numpy(dtype=float)

    finite = np.isfinite(dt_model) & np.isfinite(dt_geom) & np.isfinite(dt_resid)

    max_abs_resid = float(np.nanmax(abs_resid)) if len(abs_resid) else np.nan
    p95_abs_resid = float(np.nanquantile(abs_resid, 0.95)) if len(abs_resid) else np.nan
    median_abs_resid = float(np.nanmedian(abs_resid)) if len(abs_resid) else np.nan

    row0 = g.iloc[0]

    summary_rows.append({
        "param_set": param_set,
        "anchor_label": anchor_label,
        "phase_label": phase_label,

        "n_rows_segment_A": len(g),
        "n_finite": int(finite.sum()),
        "n_nonfinite": int((~finite).sum()),

        "Q_lo_segment_A_Ah": float(row0["segment_A_Q_lo_Ah"]),
        "Q_hi_segment_A_Ah": float(row0["segment_A_Q_hi_Ah"]),
        "Q_min_Ah": float(g["Q_Ah"].min()),
        "Q_max_Ah": float(g["Q_Ah"].max()),
        "segment_A_before_earliest_Vmax": bool(row0["segment_A_before_earliest_Vmax"]),

        "Q_to_Vmax_DC_Ah": float(row0["Q_to_Vmax_DC_Ah"]),
        "Q_to_Vmax_DCAC_Ah": float(row0["Q_to_Vmax_DCAC_Ah"]),
        "Q_to_Vmax_shift_Ah": float(row0["Q_to_Vmax_shift_Ah"]),
        "Q_to_Vmax_shift_pct_nom": float(row0["Q_to_Vmax_shift_pct_nom"]),
        "t_to_Vmax_shift_s": float(row0["t_to_Vmax_shift_s"]),
        "has_cv_feedback_in_current_npz": bool(row0["has_cv_feedback_in_current_npz"]),

        "dt_model_mean_s": float(np.nanmean(dt_model)),
        "dt_model_median_s": float(np.nanmedian(dt_model)),
        "dt_model_min_s": float(np.nanmin(dt_model)),
        "dt_model_max_s": float(np.nanmax(dt_model)),
        "dt_model_sign_topology": sign_topology(dt_model),

        "dt_geom_mean_s": float(np.nanmean(dt_geom)),
        "dt_geom_median_s": float(np.nanmedian(dt_geom)),
        "dt_geom_min_s": float(np.nanmin(dt_geom)),
        "dt_geom_max_s": float(np.nanmax(dt_geom)),
        "dt_geom_sign_topology": sign_topology(dt_geom),

        "dt_resid_mean_s": float(np.nanmean(dt_resid)),
        "dt_resid_median_s": float(np.nanmedian(dt_resid)),
        "dt_resid_min_s": float(np.nanmin(dt_resid)),
        "dt_resid_max_s": float(np.nanmax(dt_resid)),
        "dt_resid_mean_abs_s": float(np.nanmean(abs_resid)),
        "dt_resid_median_abs_s": median_abs_resid,
        "dt_resid_p95_abs_s": p95_abs_resid,
        "dt_resid_max_abs_s": max_abs_resid,
        "dt_resid_sign_topology": sign_topology(dt_resid),

        "resid_over_geom_median": float(np.nanmedian(resid_over_geom)),
        "resid_over_geom_p95": float(np.nanquantile(resid_over_geom, 0.95)),
        "resid_over_geom_max": float(np.nanmax(resid_over_geom)),

        "resid_over_model_median": float(np.nanmedian(resid_over_model)),
        "resid_over_model_p95": float(np.nanquantile(resid_over_model, 0.95)),
        "resid_over_model_max": float(np.nanmax(resid_over_model)),

        "dtQ_identity_err_max_abs_s": float(np.nanmax(segA["dtQ_identity_err_s"].abs())),

        "Q_nom_Ah": float(row0["Q_nom_Ah"]),
        "f_anchor_Hz": float(row0["f_anchor_Hz"]),
        "T_period_s": float(row0["T_period_s"]),

        "segment_A_status": segment_A_classification(
            max_abs_resid=max_abs_resid,
            p95_abs_resid=p95_abs_resid,
            median_abs_resid=median_abs_resid,
        ),

        "interpretation": (
            "Segment A is prescribed-current geometry-dominated"
            if max_abs_resid < 1.0
            else "Segment A residual requires inspection"
        ),
    })

summary_df = pd.DataFrame(summary_rows)

# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_curves = DATA / "day20_step3_segment_A_geometry_residual_curves.csv"
out_summary = DATA / "day20_step3_segment_A_geometry_residual_summary.csv"

segA.to_csv(out_curves, index=False)
summary_df.to_csv(out_summary, index=False)

print(f"Wrote: {out_curves} ({len(segA)} rows)")
print(f"Wrote: {out_summary} ({len(summary_df)} rows)")

display(summary_df)

print("\nSegment A status counts:")
print(summary_df["segment_A_status"].value_counts(dropna=False).to_string())

print("\nKey Segment A metrics:")
display(
    summary_df[
        [
            "param_set",
            "dt_model_median_s",
            "dt_geom_median_s",
            "dt_resid_median_s",
            "dt_resid_p95_abs_s",
            "dt_resid_max_abs_s",
            "resid_over_geom_median",
            "Q_to_Vmax_shift_Ah",
            "t_to_Vmax_shift_s",
            "segment_A_status",
            "has_cv_feedback_in_current_npz",
        ]
    ]
)

# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert summary_df["segment_A_before_earliest_Vmax"].all(), (
    "Segment A is not entirely before earliest Vmax for some pair."
)

assert not summary_df["has_cv_feedback_in_current_npz"].any(), (
    "Unexpected CV feedback detected in current v4 trajectory cache."
)

assert (summary_df["n_nonfinite"] == 0).all(), (
    "Non-finite Segment A values detected:\n"
    f"{summary_df[summary_df['n_nonfinite'] != 0][['param_set', 'n_nonfinite']]}"
)

assert (summary_df["dtQ_identity_err_max_abs_s"] < 1e-9).all(), (
    "dtQ_s != dtQ_geom_s + dtQ_resid_s identity violation."
)

print("\n" + "=" * 72)
print("Cell 5 PASSED — Segment A geometry/residual audit complete")
print("=" * 72)

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step3_segment_A_geometry_residual_curves.csv (183 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step3_segment_A_geometry_residual_summary.csv (3 rows)


,param_set,anchor_label,phase_label,n_rows_segment_A,n_finite,n_nonfinite,Q_lo_segment_A_Ah,Q_hi_segment_A_Ah,Q_min_Ah,Q_max_Ah,...,resid_over_geom_max,resid_over_model_median,resid_over_model_p95,resid_over_model_max,dtQ_identity_err_max_abs_s,Q_nom_Ah,f_anchor_Hz,T_period_s,segment_A_status,interpretation
0,Chen2020,Route2_AC0p5C_10tau_native_charge_first,charge_first,62,62,0,1.0,3.668944,1.029000,3.668944,...,0.000009,1.516077e-06,0.000005,0.000009,4.547474e-13,5.0,0.000408,2453.513699,numerical_null,Segment A is prescribed-current geometry-domin...
1,OKane2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,62,62,0,1.0,3.686822,1.033073,3.686822,...,0.000008,1.114545e-06,0.000003,0.000008,4.547474e-13,5.0,0.000397,2516.435063,numerical_null,Segment A is prescribed-current geometry-domin...
2,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,59,59,0,1.0,3.110457,1.010375,3.110457,...,0.000006,4.892002e-07,0.000003,0.000006,4.547474e-13,5.0,0.000333,2998.832183,numerical_null,Segment A is prescribed-current geometry-domin...



Segment A status counts:
segment_A_status
numerical_null    3

Key Segment A metrics:


,param_set,dt_model_median_s,dt_geom_median_s,dt_resid_median_s,dt_resid_p95_abs_s,dt_resid_max_abs_s,resid_over_geom_median,Q_to_Vmax_shift_Ah,t_to_Vmax_shift_s,segment_A_status,has_cv_feedback_in_current_npz
0,Chen2020,1302.892568,1302.894455,-0.001338,0.007722,0.014259,1.516074e-06,0.856056,3852.061493,numerical_null,False
1,OKane2022,1345.949033,1345.949270,-0.000882,0.003514,0.005144,1.114545e-06,0.807622,3497.993316,numerical_null,False
2,ORegan2022,1378.560563,1378.560482,0.000279,0.005973,0.014275,4.892004e-07,1.513671,7081.007881,numerical_null,False



Cell 5 PASSED — Segment A geometry/residual audit complete


## Segment B — voltage-boundary event separation

Segment A has shown that the charge-first prescribed-current shared-Q window is geometry-dominated:

$$
\Delta t_{\mathrm{model}}(Q)
\approx
\Delta t_{\mathrm{geom}}(Q),
\qquad
\Delta t_{\mathrm{resid}}(Q)
\approx
0
$$

The next question is whether DCAC reaches the voltage boundary substantially earlier than the DC reference.

This cell audits only the voltage-boundary event separation. It does not include AC-off transition or CV feedback.

For later full-protocol simulations, the MJ1 experimental cutoff current must not be transferred as an absolute 50 mA value. It must be normalized as

$$
C_{\mathrm{cutoff}}
=
\frac{0.05}{3.4}
\approx
0.0147C
$$

and applied to each PyBaMM parameter set as

$$
I_{\mathrm{cutoff}}
=
C_{\mathrm{cutoff}}
\cdot
Q_{\mathrm{nom}}
$$

In [8]:
# Cell 6 — Segment B voltage-boundary contribution audit
#
# Purpose:
#   Quantify Segment B:
#       voltage-boundary event separation interval
#
# Segment B is defined by:
#       Q_to_Vmax_DCAC  →  Q_to_Vmax_DC
#
# Important:
#   Current v4 trajectory cache terminates at Vmax and contains no CV feedback tail.
#   Therefore Segment B is not a full t_DCAC(Q) curve beyond Vmax.
#   It is an event-boundary separation audit:
#
#       - At Q where DCAC reaches Vmax, where is DC?
#       - How much Q and time remain before DC reaches Vmax?
#       - How large is the event-level boundary shift?
#
# Inputs:
#   data/day18_step1_phase_audit_trajectories_v4_charge_first.npz
#   data/day20_step2_v4_event_boundary_audit.csv
#   data/day20_step2_v4_segment_boundaries.csv
#
# Output:
#   data/day20_step4_segment_B_voltage_boundary_audit.csv

import numpy as np
import pandas as pd

TRAJ_V4 = DATA / "day18_step1_phase_audit_trajectories_v4_charge_first.npz"
EVENT_AUDIT = DATA / "day20_step2_v4_event_boundary_audit.csv"
SEGMENTS = DATA / "day20_step2_v4_segment_boundaries.csv"

assert TRAJ_V4.exists(), f"Missing: {TRAJ_V4}"
assert EVENT_AUDIT.exists(), f"Missing: {EVENT_AUDIT}"
assert SEGMENTS.exists(), f"Missing: {SEGMENTS}"

z = np.load(TRAJ_V4, allow_pickle=True)
event_df = pd.read_csv(EVENT_AUDIT)
segments = pd.read_csv(SEGMENTS)

print(f"Loaded trajectory cache: {TRAJ_V4.name}")
print(f"Loaded event audit      : {EVENT_AUDIT.name} shape={event_df.shape}")
print(f"Loaded segment table    : {SEGMENTS.name} shape={segments.shape}")


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def get_traj(param_set, protocol):
    keys = {
        "t": f"{param_set}__{protocol}__t",
        "V": f"{param_set}__{protocol}__V",
        "I": f"{param_set}__{protocol}__I",
        "Q": f"{param_set}__{protocol}__Q_net",
    }

    missing = [k for k in keys.values() if k not in z.files]
    if missing:
        raise KeyError(f"Missing trajectory keys for {param_set}/{protocol}: {missing}")

    out = {
        "t_s": np.asarray(z[keys["t"]], dtype=float),
        "V_V": np.asarray(z[keys["V"]], dtype=float),
        "I_A": np.asarray(z[keys["I"]], dtype=float),
        "Q_Ah": np.asarray(z[keys["Q"]], dtype=float),
    }

    n = len(out["t_s"])
    for name, arr in out.items():
        if len(arr) != n:
            raise ValueError(f"{param_set}/{protocol}: length mismatch for {name}")
        if not np.all(np.isfinite(arr)):
            raise ValueError(f"{param_set}/{protocol}: non-finite values in {name}")

    if n > 1 and not np.all(np.diff(out["t_s"]) > 0):
        raise ValueError(f"{param_set}/{protocol}: t_s not strictly increasing")

    return out


def first_passage_by_Q(t_s, Q_Ah, target_Q_Ah, extra_arrays=None):
    """
    Raw first-passage interpolation by Q.

    Returns t(Q*) and interpolated extra variables, e.g. V(Q*), I(Q*).
    """
    t_s = np.asarray(t_s, dtype=float)
    Q_Ah = np.asarray(Q_Ah, dtype=float)

    if extra_arrays is None:
        extra_arrays = {}

    crossed = Q_Ah >= target_Q_Ah

    if not crossed.any():
        out = {
            "found": False,
            "idx": np.nan,
            "t_s": np.nan,
            "Q_Ah": np.nan,
        }
        for name in extra_arrays:
            out[name] = np.nan
        return out

    idx = int(np.argmax(crossed))

    if idx == 0:
        out = {
            "found": True,
            "idx": idx,
            "t_s": float(t_s[0]),
            "Q_Ah": float(Q_Ah[0]),
        }
        for name, arr in extra_arrays.items():
            arr = np.asarray(arr, dtype=float)
            out[name] = float(arr[0])
        return out

    q0, q1 = Q_Ah[idx - 1], Q_Ah[idx]
    t0, t1 = t_s[idx - 1], t_s[idx]

    if q1 == q0:
        frac = 1.0
    else:
        frac = (target_Q_Ah - q0) / (q1 - q0)
        frac = float(np.clip(frac, 0.0, 1.0))

    out = {
        "found": True,
        "idx": idx,
        "t_s": float(t0 + frac * (t1 - t0)),
        "Q_Ah": float(target_Q_Ah),
    }

    for name, arr in extra_arrays.items():
        arr = np.asarray(arr, dtype=float)
        y0, y1 = arr[idx - 1], arr[idx]
        out[name] = float(y0 + frac * (y1 - y0))

    return out


# ---------------------------------------------------------------------
# Validate scope from Cell 4
# ---------------------------------------------------------------------

assert not event_df["has_cv_like_current_decay"].any(), (
    "Current v4 trajectory cache unexpectedly contains CV-like current decay."
)

assert not segments["has_cv_feedback_in_current_npz"].any(), (
    "Segment table reports CV feedback available, but Day20 current scope assumes no CV tail."
)


# ---------------------------------------------------------------------
# Segment B audit
# ---------------------------------------------------------------------

rows = []

for _, seg in segments.iterrows():
    param_set = seg["param_set"]
    protocol_DC = seg["protocol_DC"]
    protocol_DCAC = seg["protocol_DCAC"]

    dc = get_traj(param_set, protocol_DC)
    dcac = get_traj(param_set, protocol_DCAC)

    Q_vmax_DC = float(seg["Q_to_Vmax_DC_Ah"])
    Q_vmax_DCAC = float(seg["Q_to_Vmax_DCAC_Ah"])
    t_vmax_DC = float(seg["t_to_Vmax_DC_s"])
    t_vmax_DCAC = float(seg["t_to_Vmax_DCAC_s"])

    Vmax_DC = float(seg["Vmax_cutoff_DC"])
    Vmax_DCAC = float(seg["Vmax_cutoff_DCAC"])

    Q_nom_Ah = float(seg["Q_nom_Ah"])

    # State of DC arm at the Q where DCAC reaches Vmax
    dc_at_Q_dcac_vmax = first_passage_by_Q(
        t_s=dc["t_s"],
        Q_Ah=dc["Q_Ah"],
        target_Q_Ah=Q_vmax_DCAC,
        extra_arrays={
            "V_DC_at_Q_DCAC_Vmax_V": dc["V_V"],
            "I_DC_at_Q_DCAC_Vmax_A": dc["I_A"],
        },
    )

    # State of DCAC arm at its own Vmax Q, reconstructed from its own trajectory
    dcac_at_Q_dcac_vmax = first_passage_by_Q(
        t_s=dcac["t_s"],
        Q_Ah=dcac["Q_Ah"],
        target_Q_Ah=Q_vmax_DCAC,
        extra_arrays={
            "V_DCAC_at_Q_DCAC_Vmax_V": dcac["V_V"],
            "I_DCAC_at_Q_DCAC_Vmax_A": dcac["I_A"],
        },
    )

    # Decompose event timing shift:
    # total_event_shift =
    #   state_equivalent_gain_at_DCAC_boundary
    #   + DC_remaining_time_from_Q_DCAC_Vmax_to_Q_DC_Vmax
    dt_state_at_DCAC_boundary_s = dc_at_Q_dcac_vmax["t_s"] - t_vmax_DCAC
    dc_remaining_time_to_own_Vmax_s = t_vmax_DC - dc_at_Q_dcac_vmax["t_s"]
    total_event_shift_s = t_vmax_DC - t_vmax_DCAC

    identity_err_s = total_event_shift_s - (
        dt_state_at_DCAC_boundary_s + dc_remaining_time_to_own_Vmax_s
    )

    Q_shift_Ah = Q_vmax_DC - Q_vmax_DCAC
    Q_shift_pct_nom = Q_shift_Ah / Q_nom_Ah * 100.0

    V_DC_at_Q_DCAC = dc_at_Q_dcac_vmax["V_DC_at_Q_DCAC_Vmax_V"]
    voltage_headroom_DC_V = Vmax_DC - V_DC_at_Q_DCAC

    # DC current estimate around the same-Q point
    I_DC_at_Q_DCAC = abs(dc_at_Q_dcac_vmax["I_DC_at_Q_DCAC_Vmax_A"])
    if np.isfinite(I_DC_at_Q_DCAC) and I_DC_at_Q_DCAC > 0:
        dc_remaining_time_from_Q_shift_constant_I_s = Q_shift_Ah * 3600.0 / I_DC_at_Q_DCAC
    else:
        dc_remaining_time_from_Q_shift_constant_I_s = np.nan

    constant_I_time_err_s = dc_remaining_time_to_own_Vmax_s - dc_remaining_time_from_Q_shift_constant_I_s

    rows.append({
        "param_set": param_set,
        "anchor_label": seg["anchor_label"],
        "phase_label": seg["phase_label"],
        "protocol_DC": protocol_DC,
        "protocol_DCAC": protocol_DCAC,

        "Q_nom_Ah": Q_nom_Ah,
        "f_anchor_Hz": float(seg["f_anchor_Hz"]),
        "T_period_s": float(seg["T_period_s"]),

        "Vmax_cutoff_DC": Vmax_DC,
        "Vmax_cutoff_DCAC": Vmax_DCAC,

        "Q_to_Vmax_DC_Ah": Q_vmax_DC,
        "Q_to_Vmax_DCAC_Ah": Q_vmax_DCAC,
        "Q_to_Vmax_shift_Ah": Q_shift_Ah,
        "Q_to_Vmax_shift_pct_nom": Q_shift_pct_nom,

        "t_to_Vmax_DC_s": t_vmax_DC,
        "t_to_Vmax_DCAC_s": t_vmax_DCAC,
        "t_to_Vmax_shift_s": total_event_shift_s,

        "DC_at_Q_DCAC_Vmax_found": dc_at_Q_dcac_vmax["found"],
        "t_DC_at_Q_DCAC_Vmax_s": dc_at_Q_dcac_vmax["t_s"],
        "V_DC_at_Q_DCAC_Vmax_V": V_DC_at_Q_DCAC,
        "I_DC_at_Q_DCAC_Vmax_A": dc_at_Q_dcac_vmax["I_DC_at_Q_DCAC_Vmax_A"],

        "DCAC_at_Q_DCAC_Vmax_found": dcac_at_Q_dcac_vmax["found"],
        "t_DCAC_at_Q_DCAC_Vmax_s_reconstructed": dcac_at_Q_dcac_vmax["t_s"],
        "V_DCAC_at_Q_DCAC_Vmax_V_reconstructed": dcac_at_Q_dcac_vmax["V_DCAC_at_Q_DCAC_Vmax_V"],
        "I_DCAC_at_Q_DCAC_Vmax_A_reconstructed": dcac_at_Q_dcac_vmax["I_DCAC_at_Q_DCAC_Vmax_A"],

        "voltage_headroom_DC_at_DCAC_Vmax_Q_V": voltage_headroom_DC_V,

        "dt_state_at_DCAC_boundary_s": dt_state_at_DCAC_boundary_s,
        "dc_remaining_time_to_own_Vmax_s": dc_remaining_time_to_own_Vmax_s,
        "event_shift_decomposition_identity_err_s": identity_err_s,

        "dc_remaining_time_from_Q_shift_constant_I_s": dc_remaining_time_from_Q_shift_constant_I_s,
        "constant_I_time_err_s": constant_I_time_err_s,

        "segment_A_Q_lo_Ah": float(seg["segment_A_Q_lo_Ah"]),
        "segment_A_Q_hi_Ah": float(seg["segment_A_Q_hi_Ah"]),
        "segment_A_before_earliest_Vmax": bool(seg["segment_A_before_earliest_Vmax"]),

        "segment_B_Q_lo_Ah": float(seg["segment_B_Q_lo_Ah"]),
        "segment_B_Q_hi_Ah": float(seg["segment_B_Q_hi_Ah"]),
        "segment_B_span_Ah": float(seg["segment_B_span_Ah"]),
        "segment_B_available_in_current_npz": bool(seg["segment_B_available_in_current_npz"]),
        "has_cv_feedback_in_current_npz": bool(seg["has_cv_feedback_in_current_npz"]),

        "interpretation": (
            "large voltage-boundary shift; current NPZ stops at Vmax and does not contain CV feedback"
        ),
    })

boundary_df = pd.DataFrame(rows)

# Verdict flags
boundary_df["Q_shift_positive"] = boundary_df["Q_to_Vmax_shift_Ah"] > 0
boundary_df["t_shift_positive"] = boundary_df["t_to_Vmax_shift_s"] > 0
boundary_df["DC_below_Vmax_at_DCAC_boundary"] = boundary_df["voltage_headroom_DC_at_DCAC_Vmax_Q_V"] > 0

boundary_df["boundary_shift_class"] = np.select(
    [
        boundary_df["Q_to_Vmax_shift_pct_nom"] >= 20.0,
        boundary_df["Q_to_Vmax_shift_pct_nom"] >= 10.0,
        boundary_df["Q_to_Vmax_shift_pct_nom"] > 0.0,
    ],
    [
        "very_large_boundary_shift",
        "large_boundary_shift",
        "positive_boundary_shift",
    ],
    default="no_positive_boundary_shift",
)

out = DATA / "day20_step4_segment_B_voltage_boundary_audit.csv"
boundary_df.to_csv(out, index=False)

print(f"Wrote: {out}")
display(boundary_df)

print("\nSegment B boundary shift summary:")
display(
    boundary_df[
        [
            "param_set",
            "Q_to_Vmax_shift_Ah",
            "Q_to_Vmax_shift_pct_nom",
            "t_to_Vmax_shift_s",
            "dt_state_at_DCAC_boundary_s",
            "dc_remaining_time_to_own_Vmax_s",
            "voltage_headroom_DC_at_DCAC_Vmax_Q_V",
            "boundary_shift_class",
            "has_cv_feedback_in_current_npz",
        ]
    ]
)

# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert boundary_df["Q_shift_positive"].all(), (
    "Expected DC to reach Vmax at higher Q than DCAC for all v4 pairs."
)

assert boundary_df["t_shift_positive"].all(), (
    "Expected DC to reach Vmax later than DCAC for all v4 pairs."
)

assert boundary_df["DC_below_Vmax_at_DCAC_boundary"].all(), (
    "Expected DC to remain below Vmax at Q where DCAC reaches Vmax."
)

assert (boundary_df["event_shift_decomposition_identity_err_s"].abs() < 1e-6).all(), (
    "Event shift decomposition identity failed:\n"
    f"{boundary_df[['param_set', 'event_shift_decomposition_identity_err_s']]}"
)

assert not boundary_df["has_cv_feedback_in_current_npz"].any(), (
    "Current v4 NPZ unexpectedly contains CV feedback."
)

print("\n" + "=" * 72)
print("Cell 6 PASSED — Segment B voltage-boundary contribution audit complete")
print("=" * 72)

Loaded trajectory cache: day18_step1_phase_audit_trajectories_v4_charge_first.npz
Loaded event audit      : day20_step2_v4_event_boundary_audit.csv shape=(6, 34)
Loaded segment table    : day20_step2_v4_segment_boundaries.csv shape=(3, 28)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day20_step4_segment_B_voltage_boundary_audit.csv


,param_set,anchor_label,phase_label,protocol_DC,protocol_DCAC,Q_nom_Ah,f_anchor_Hz,T_period_s,Vmax_cutoff_DC,Vmax_cutoff_DCAC,...,segment_B_Q_lo_Ah,segment_B_Q_hi_Ah,segment_B_span_Ah,segment_B_available_in_current_npz,has_cv_feedback_in_current_npz,interpretation,Q_shift_positive,t_shift_positive,DC_below_Vmax_at_DCAC_boundary,boundary_shift_class
0,Chen2020,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,5.0,0.000408,2453.513699,4.2,4.2,...,3.768944,4.625000,0.856056,False,False,large voltage-boundary shift; current NPZ stop...,True,True,True,large_boundary_shift
1,OKane2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,5.0,0.000397,2516.435063,4.2,4.2,...,3.786822,4.594444,0.807622,False,False,large voltage-boundary shift; current NPZ stop...,True,True,True,large_boundary_shift
2,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,DC_0p2C,DCAC_DC0p2C_AC0p5C_10tau,5.0,0.000333,2998.832183,4.4,4.4,...,3.210457,4.724128,1.513671,False,False,large voltage-boundary shift; current NPZ stop...,True,True,True,very_large_boundary_shift



Segment B boundary shift summary:


,param_set,Q_to_Vmax_shift_Ah,Q_to_Vmax_shift_pct_nom,t_to_Vmax_shift_s,dt_state_at_DCAC_boundary_s,dc_remaining_time_to_own_Vmax_s,voltage_headroom_DC_at_DCAC_Vmax_Q_V,boundary_shift_class,has_cv_feedback_in_current_npz
0,Chen2020,0.856056,17.121118,3852.061493,770.260282,3081.801211,0.113419,large_boundary_shift,False
1,OKane2022,0.807622,16.152443,3497.993316,590.553554,2907.439762,0.106144,large_boundary_shift,False
2,ORegan2022,1.513671,30.273420,7081.007881,1631.792248,5449.215633,0.454153,very_large_boundary_shift,False



Cell 6 PASSED — Segment B voltage-boundary contribution audit complete


In [9]:
# Cell 7 — Day20A interim summary: Segment A/B pre-CV audit
#
# Purpose:
#   Close Day20A pre-CV / event-boundary audit.
#
# Inputs:
#   data/day20_step2_v4_event_boundary_audit.csv
#   data/day20_step2_v4_segment_boundaries.csv
#   data/day20_step3_segment_A_geometry_residual_summary.csv
#   data/day20_step4_segment_B_voltage_boundary_audit.csv
#
# Outputs:
#   data/day20_step5_interim_segment_AB_summary.csv
#   docs/day20A_preCV_event_boundary_audit.md
#
# Scope:
#   This closes the current existing-trajectory audit only.
#   It does NOT claim full-protocol CC+CV behavior because current NPZ
#   terminates at Vmax and contains no CV feedback tail.

import numpy as np
import pandas as pd
from pathlib import Path

EVENT_AUDIT = DATA / "day20_step2_v4_event_boundary_audit.csv"
SEGMENTS = DATA / "day20_step2_v4_segment_boundaries.csv"
SEG_A = DATA / "day20_step3_segment_A_geometry_residual_summary.csv"
SEG_B = DATA / "day20_step4_segment_B_voltage_boundary_audit.csv"

for p in [EVENT_AUDIT, SEGMENTS, SEG_A, SEG_B]:
    assert p.exists(), f"Missing required Day20A file: {p}"

event_df = pd.read_csv(EVENT_AUDIT)
segments_df = pd.read_csv(SEGMENTS)
segA_df = pd.read_csv(SEG_A)
segB_df = pd.read_csv(SEG_B)

# ---------------------------------------------------------------------
# Build pair-level interim summary
# ---------------------------------------------------------------------

join_keys = ["param_set", "anchor_label", "phase_label"]

merged = segA_df.merge(
    segB_df,
    on=join_keys,
    how="inner",
    suffixes=("_segA", "_segB"),
    validate="one_to_one",
)

assert len(merged) == len(segA_df) == len(segB_df), (
    f"Expected one-to-one merge between Segment A and B summaries; "
    f"got segA={len(segA_df)}, segB={len(segB_df)}, merged={len(merged)}"
)

rows = []

for _, r in merged.iterrows():
    param_set = r["param_set"]
    
    # Segment A metrics
    segA_status = r["segment_A_status"]
    segA_resid_max = float(r["dt_resid_max_abs_s"])
    segA_resid_p95 = float(r["dt_resid_p95_abs_s"])
    segA_resid_median = float(r["dt_resid_median_s"])
    segA_geom_median = float(r["dt_geom_median_s"])
    segA_model_median = float(r["dt_model_median_s"])
    resid_over_geom_median = float(r["resid_over_geom_median"])
    
    # Segment B metrics
    q_shift_Ah = float(r["Q_to_Vmax_shift_Ah_segB"])
    q_shift_pct = float(r["Q_to_Vmax_shift_pct_nom"])
    t_shift_s = float(r["t_to_Vmax_shift_s_segB"])
    dt_state_boundary = float(r["dt_state_at_DCAC_boundary_s"])
    dc_remaining = float(r["dc_remaining_time_to_own_Vmax_s"])
    voltage_headroom = float(r["voltage_headroom_DC_at_DCAC_Vmax_Q_V"])
    boundary_class = r["boundary_shift_class"]
    
    # Contribution fractions for event timing shift
    if np.isfinite(t_shift_s) and abs(t_shift_s) > 1e-12:
        state_gain_fraction = dt_state_boundary / t_shift_s
        dc_remaining_fraction = dc_remaining / t_shift_s
    else:
        state_gain_fraction = np.nan
        dc_remaining_fraction = np.nan
    
    # Scope flags
    has_cv_feedback = bool(r["has_cv_feedback_in_current_npz_segB"])
    
    # Interpretive verdict
    if segA_status == "numerical_null" and q_shift_Ah > 0 and not has_cv_feedback:
        day20A_verdict = "segment_A_geometry_dominated_segment_B_boundary_shift_large_no_CV_tail"
    elif segA_status == "numerical_null" and q_shift_Ah > 0 and has_cv_feedback:
        day20A_verdict = "segment_A_geometry_dominated_boundary_shift_large_with_CV_tail_available"
    else:
        day20A_verdict = "requires_review"
    
    rows.append({
        "param_set": param_set,
        "anchor_label": r["anchor_label"],
        "phase_label": r["phase_label"],
        
        "protocol_DC": r["protocol_DC"],
        "protocol_DCAC": r["protocol_DCAC"],
        "Q_nom_Ah": float(r["Q_nom_Ah_segB"]),
        "f_anchor_Hz": float(r["f_anchor_Hz_segB"]),
        "T_period_s": float(r["T_period_s_segB"]),
        
        # Segment A
        "segment_A_status": segA_status,
        "segment_A_n_rows": int(r["n_rows_segment_A"]),
        "segment_A_Q_lo_Ah": float(r["Q_lo_segment_A_Ah"]),
        "segment_A_Q_hi_Ah": float(r["Q_hi_segment_A_Ah"]),
        "segment_A_dt_model_median_s": segA_model_median,
        "segment_A_dt_geom_median_s": segA_geom_median,
        "segment_A_dt_resid_median_s": segA_resid_median,
        "segment_A_dt_resid_p95_abs_s": segA_resid_p95,
        "segment_A_dt_resid_max_abs_s": segA_resid_max,
        "segment_A_resid_over_geom_median": resid_over_geom_median,
        
        # Segment B
        "Q_to_Vmax_DC_Ah": float(r["Q_to_Vmax_DC_Ah_segB"]),
        "Q_to_Vmax_DCAC_Ah": float(r["Q_to_Vmax_DCAC_Ah_segB"]),
        "Q_to_Vmax_shift_Ah": q_shift_Ah,
        "Q_to_Vmax_shift_pct_nom": q_shift_pct,
        "t_to_Vmax_DC_s": float(r["t_to_Vmax_DC_s"]),
        "t_to_Vmax_DCAC_s": float(r["t_to_Vmax_DCAC_s"]),
        "t_to_Vmax_shift_s": t_shift_s,
        "dt_state_at_DCAC_boundary_s": dt_state_boundary,
        "dc_remaining_time_to_own_Vmax_s": dc_remaining,
        "state_gain_fraction_of_event_shift": state_gain_fraction,
        "dc_remaining_fraction_of_event_shift": dc_remaining_fraction,
        "voltage_headroom_DC_at_DCAC_Vmax_Q_V": voltage_headroom,
        "boundary_shift_class": boundary_class,
        
        # Scope
        "has_cv_feedback_in_current_npz": has_cv_feedback,
        "current_npz_scope": "pre_CV_event_boundary_only",
        "day20A_verdict": day20A_verdict,
        "next_required_step": (
            "new full CC+CV PyBaMM protocol with AC-off at Vmax and normalized cutoff current"
            if not has_cv_feedback else
            "audit available CV feedback tail"
        ),
    })

summary_df = pd.DataFrame(rows)

out_summary = DATA / "day20_step5_interim_segment_AB_summary.csv"
summary_df.to_csv(out_summary, index=False)

print(f"Wrote: {out_summary} ({len(summary_df)} rows)")
display(summary_df)

print("\nDay20A verdict counts:")
print(summary_df["day20A_verdict"].value_counts(dropna=False).to_string())

print("\nKey compact summary:")
display(
    summary_df[
        [
            "param_set",
            "segment_A_status",
            "segment_A_dt_resid_p95_abs_s",
            "segment_A_dt_resid_max_abs_s",
            "Q_to_Vmax_shift_Ah",
            "Q_to_Vmax_shift_pct_nom",
            "t_to_Vmax_shift_s",
            "state_gain_fraction_of_event_shift",
            "dc_remaining_fraction_of_event_shift",
            "voltage_headroom_DC_at_DCAC_Vmax_Q_V",
            "boundary_shift_class",
            "has_cv_feedback_in_current_npz",
            "day20A_verdict",
        ]
    ]
)

# ---------------------------------------------------------------------
# Write Markdown note
# ---------------------------------------------------------------------

DOCS.mkdir(exist_ok=True)
out_doc = DOCS / "day20A_preCV_event_boundary_audit.md"

lines = []
lines.append("# Day20A — PyBaMM pre-CV / voltage-boundary segmented audit")
lines.append("")
lines.append("Status: interim audit closed")
lines.append("")
lines.append("## Scope")
lines.append("")
lines.append("Day20A audits the existing Day18 v4 charge-first PyBaMM trajectory cache:")
lines.append("")
lines.append("```text")
lines.append("data/day18_step1_phase_audit_trajectories_v4_charge_first.npz")
lines.append("```")
lines.append("")
lines.append("The cache contains `t`, `V`, `I`, and `Q_net` for DC and DCAC arms, but it terminates at the maximum-voltage event. It does not contain an AC-off transition or a CV feedback tail.")
lines.append("")
lines.append("Therefore Day20A covers:")
lines.append("")
lines.append("- Segment A: AC-on prescribed-current shared-Q window")
lines.append("- Segment B: voltage-boundary event separation")
lines.append("")
lines.append("Day20A does not cover:")
lines.append("")
lines.append("- Segment C: AC-off transition")
lines.append("- Segment D: pure DC-CV feedback to cutoff")
lines.append("")
lines.append("## Segment A finding")
lines.append("")
lines.append("Segment A is geometry-dominated for all audited parameter sets.")
lines.append("")
lines.append("| param_set | median Δt_model [s] | median Δt_geom [s] | median Δt_resid [s] | p95 |Δt_resid| [s] | max |Δt_resid| [s] |")
lines.append("|---|---:|---:|---:|---:|---:|")

for _, r in summary_df.iterrows():
    lines.append(
        f"| {r['param_set']} | "
        f"{r['segment_A_dt_model_median_s']:.3f} | "
        f"{r['segment_A_dt_geom_median_s']:.3f} | "
        f"{r['segment_A_dt_resid_median_s']:.6f} | "
        f"{r['segment_A_dt_resid_p95_abs_s']:.6f} | "
        f"{r['segment_A_dt_resid_max_abs_s']:.6f} |"
    )

lines.append("")
lines.append("Interpretation:")
lines.append("")
lines.append("```text")
lines.append("Segment A raw Δt(Q) is explained by current geometry. Non-geometric residual is numerical-null.")
lines.append("```")
lines.append("")
lines.append("## Segment B finding")
lines.append("")
lines.append("Segment B shows large voltage-boundary separation. DCAC reaches the voltage boundary at substantially lower Q and earlier time than DC.")
lines.append("")
lines.append("| param_set | Q shift [Ah] | Q shift [% nominal] | time shift [s] | state gain at DCAC boundary [s] | DC remaining time [s] | DC voltage headroom [V] | class |")
lines.append("|---|---:|---:|---:|---:|---:|---:|---|")

for _, r in summary_df.iterrows():
    lines.append(
        f"| {r['param_set']} | "
        f"{r['Q_to_Vmax_shift_Ah']:.6f} | "
        f"{r['Q_to_Vmax_shift_pct_nom']:.3f} | "
        f"{r['t_to_Vmax_shift_s']:.3f} | "
        f"{r['dt_state_at_DCAC_boundary_s']:.3f} | "
        f"{r['dc_remaining_time_to_own_Vmax_s']:.3f} | "
        f"{r['voltage_headroom_DC_at_DCAC_Vmax_Q_V']:.6f} | "
        f"{r['boundary_shift_class']} |"
    )

lines.append("")
lines.append("The event timing shift decomposes as:")
lines.append("")
lines.append("```text")
lines.append("t_DC,Vmax − t_DCAC,Vmax")
lines.append("=")
lines.append("[t_DC(Q_DCAC,Vmax) − t_DCAC,Vmax]")
lines.append("+")
lines.append("[t_DC,Vmax − t_DC(Q_DCAC,Vmax)]")
lines.append("```")
lines.append("")
lines.append("The second term dominates: at the Q where DCAC reaches the voltage boundary, the DC arm is still below Vmax and still requires substantial additional charging time.")
lines.append("")
lines.append("## Cutoff normalization rule for future full-protocol simulations")
lines.append("")
lines.append("The MJ1 experimental cutoff current of 50 mA must not be transferred as an absolute current to all PyBaMM parameter sets.")
lines.append("")
lines.append("It must be normalized as:")
lines.append("")
lines.append("```text")
lines.append("C_cutoff = 0.05 / 3.4 ≈ 0.0147059 C")
lines.append("I_cutoff = C_cutoff · Q_nom")
lines.append("```")
lines.append("")
lines.append("## Day20A conclusion")
lines.append("")
lines.append("```text")
lines.append("The existing charge-first PyBaMM v4 trajectory cache supports a pre-CV audit only.")
lines.append("Segment A is current-geometry dominated with numerical-null residual.")
lines.append("Segment B shows large voltage-boundary separation.")
lines.append("The current cache does not contain AC-off transition or CV feedback.")
lines.append("A new full CC+CV PyBaMM protocol is required for Segment C/D.")
lines.append("```")
lines.append("")

out_doc.write_text("\n".join(lines), encoding="utf-8")

print(f"\nWrote: {out_doc}")

# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert (summary_df["segment_A_status"] == "numerical_null").all(), (
    "Segment A is not numerical-null for all parameter sets."
)

assert (summary_df["Q_to_Vmax_shift_Ah"] > 0).all(), (
    "Expected positive Q_to_Vmax shift for all parameter sets."
)

assert (summary_df["t_to_Vmax_shift_s"] > 0).all(), (
    "Expected positive t_to_Vmax shift for all parameter sets."
)

assert not summary_df["has_cv_feedback_in_current_npz"].any(), (
    "Unexpected CV feedback in current NPZ."
)

print("\n" + "=" * 72)
print("Cell 7 PASSED — Day20A interim Segment A/B summary complete")
print("=" * 72)

KeyError: 'Q_to_Vmax_shift_pct_nom'